In [ ]:
from src.utils.data_utils import build_dataloaders
from src.tasks import Tasks
from src.model import WM_Model
from src.config import Config
from src.utils.train_utils import BATCH_ADAPTERS, TASK_META_MAP

import torch
from tqdm.auto import tqdm

import numpy as np
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt
from sklearn.manifold import TSNE


In [ ]:
def load_config_from_checkpoint(config_dict):

    new_config = Config.__new__(Config)
    for key, value in config_dict.items():
        setattr(new_config, key, value)

    return new_config

def _to_device(batch: dict, device):

    """
    Moves all tensor values in the batch dict to the trainer device.
    Keeps non-tensors (ints, strings, etc.) unchanged.
    """
    out = {}
    for k, v in batch.items():
        out[k] = v.to(device) if torch.is_tensor(v) else v

    return out

In [ ]:
def run_sts_analysis(model_names):

    loader_exists = False

    for model_name in model_names:

        device = "cuda" if torch.cuda.is_available() else "cpu"

        # load checkpoint
        checkpoint_data = torch.load(f"WM_Bench/{model_name}/checkpoints/best.pt", weights_only = False)

        # set up config
        config_dict = checkpoint_data["config"]
        config = load_config_from_checkpoint(config_dict)
        config.train_config.batch_size = 1
        task = Tasks.SPATIAL_TASK_SWITCHING
        config.task_config.task_list = [task]

        # set up  model
        model = WM_Model(config, device).to(device)
        model.load_state_dict(checkpoint_data["model_state_dict"])

        # set up the test_loader
        if not loader_exists:
            loaders = build_dataloaders(config)
            sts_test_loader = loaders[Tasks.SPATIAL_TASK_SWITCHING]["test"]

            loader_exists = True

        meta = TASK_META_MAP[task]
        pad_val = meta["pad_value"]

        probe_hidden_states_list = []
        probe_gt_list = []
        task_identities_list = []

        # loop through the loader
        for batch_idx, raw_batch in enumerate(tqdm(sts_test_loader)):

            # for every batch
            batch = BATCH_ADAPTERS[task](raw_batch)
            batch = _to_device(batch, device)

            # img_seq shape ---> [1, 20, 3, 32, 32]
            # gt shape --> [1, 20]
            # seq_len shape --> [1]
            # task_order shape --> [1, 20]
            img_seq, gt, seq_len, task_order = batch["img_seq"], batch["gt"], batch["seq_len"], batch["task_order"]

            # pass the img_seq to the model
            model.eval()
            with torch.no_grad():

                # mem_output shape --> (1,20,256)
                output, mem_output, mem_h_n, projection_output, cnn_output = model(img_seq, task, seq_len)

                probe_mask = gt[0] != pad_val
                probe_hidden_states_list.append(mem_output[0][probe_mask].detach().cpu())
                probe_gt_list.append(gt[0][probe_mask].detach().cpu())
                task_identities_list.append(task_order[0][probe_mask].detach().cpu())

    probe_hidden_states = np.vstack(probe_hidden_states_list) # shape --> (N, 256)
    task_identities = np.concatenate(task_identities_list) # shape --> (N,)
    probe_gt = np.concatenate(probe_gt_list) # shape --> (N,)

    # time to run PCA
    pca = PCA(n_components= 2)
    hidden_pca = pca.fit_transform(probe_hidden_states)

    fig, axes = plt.subplots(1, 2, figsize=(14,6))

    # Left plot
    axes[0].scatter(hidden_pca[:, 0], hidden_pca[:, 1],
                    c=probe_gt,
                    cmap='coolwarm',
                    alpha = 0.3,
                    s = 1)
    axes[0].set_title('PCA — Colored by Response')
    axes[0].set_xlabel('PC1')
    axes[0].set_ylabel('PC2')

    # Right plot — colored by task identity
    axes[1].scatter(hidden_pca[:, 0], hidden_pca[:, 1],
                    c=task_identities,
                    cmap='coolwarm',
                    alpha=0.3,
                    s=1)
    axes[1].set_title('PCA — Colored by Task Identity')
    axes[1].set_xlabel('PC1')
    axes[1].set_ylabel('PC2')

    plt.suptitle(f'STS Neural Analysis — {model_name}')
    plt.tight_layout()
    plt.show()

    pca_50 = PCA(n_components = 50)
    hidden_50d = pca_50.fit_transform(probe_hidden_states)

    tsne = TSNE(n_components= 2, perplexity = 30, random_state = config.optimization_config.seed, n_iter = 1000)
    hidden_tsne =  tsne.fit_transform(hidden_50d)

    fig, axes = plt.subplots(1, 2, figsize=(14, 6))

    # Left plot — colored by response
    axes[0].scatter(hidden_tsne[:, 0], hidden_tsne[:, 1],
                    c=probe_gt,
                    cmap='coolwarm',
                    alpha=0.3,
                    s=1)
    axes[0].set_title('t-SNE — Colored by Response')
    axes[0].set_xlabel('t-SNE 1')
    axes[0].set_ylabel('t-SNE 2')

    # Right plot — colored by task identity
    axes[1].scatter(hidden_tsne[:, 0], hidden_tsne[:, 1],
                    c=task_identities,
                    cmap='coolwarm',
                    alpha=0.3,
                    s=1)
    axes[1].set_title('t-SNE — Colored by Task Identity')
    axes[1].set_xlabel('t-SNE 1')
    axes[1].set_ylabel('t-SNE 2')

    plt.suptitle(f'STS Neural Analysis t-SNE — {model_name}')
    plt.tight_layout()
    plt.show()

